
# 04c. Línea Base Transformers (selección explícita del mejor standalone)

**Objetivo:** comparar de forma homogénea los baselines Transformer en `train/dev` y seleccionar explícitamente el mejor baseline contextual standalone.
**Entradas (inputs):** `data/splits/train_denoised.csv` y `data/splits/<split>_denoised.csv`.
**Salidas (outputs):** `data/<modelo>_eval.csv`, `data/<modelo>_classification_report.csv`, `data/<modelo>_predicciones_<split>.csv`, y artefactos de selección en `data/outputs/transformer_baseline_selection_<timestamp>.json`.
**Notebook anterior:** `notebooks/pipeline/03_denoising_reglas_core.ipynb`.
**Notebook siguiente:** `notebooks/pipeline/06_ingenieria_features_hibridas.ipynb` y `notebooks/pipeline/08_resultados_hibrido_vs_lineas_base.ipynb`.

> **Uso metodológico:** este notebook decide el mejor baseline Transformer standalone en `dev`. La elección del backbone del híbrido no se resuelve aquí, sino en la comparación controlada posterior; `06` usa BETO por defecto y solo consume la salida de `04c` si se fuerza `FE_TEXT_BACKBONE=auto`.


## Técnicas, herramientas y librerías de esta etapa

- **Técnica principal:** fine-tuning o reevaluación controlada de baselines Transformer sobre texto clínico.
- **Herramientas/librerías:** `transformers`, `datasets`, `torch`, `pandas`, `numpy`, utilidades de `utils_shared` para métricas y splits.
- **Por qué es adecuada aquí:** esta etapa responde a una pregunta específica del proyecto: cuál es el mejor baseline contextual standalone en `dev`. Eso justifica experimentalmente qué backbone merece pasar a comparación posterior.
- **Limitación:** que un Transformer gane como baseline standalone no implica que sea automáticamente el mejor backbone del híbrido. Esa segunda pregunta se resuelve aparte en la comparación controlada de backbones del híbrido.
- **Alternativa/metodología complementaria:** mantener solo un Transformer por conveniencia sería más simple, pero metodológicamente más débil. Por eso aquí se comparan `BETO`, `ROBERTA_CLINICAL` y, cuando aplica, `ROBERTA_BIOMEDICAL`.


## Criterio metodológico
- Este notebook mantiene la lógica baseline de fine-tuning en texto clínico.
- En el corte actual la tarea es binaria (`ansiedad`, `depresion`) con salida probabilística.
- Incluye modo rápido opcional para auditoría técnica (`TRF_MODO_RAPIDO=1`).
- Mantiene un modo de re-evaluación (`TRF_REUSAR_PREDICCIONES=1`) para recalcular métricas sobre `<split>_denoised` sin reentrenar cuando ya existen predicciones.


In [1]:
import os
import json
import re
import shutil
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from utils_shared import setup_paths, load_splits, calculate_metrics

paths = setup_paths()
DATA_PATH = paths['DATA_PATH']
SPLITS_PATH = paths['SPLITS_PATH']
CHECKPOINTS_PATH = paths['CHECKPOINTS_PATH']
LOGS_PATH = paths['LOGS_PATH']
OUTPUTS_PATH = paths['OUTPUTS_PATH']

MODO_RAPIDO = os.getenv('TRF_MODO_RAPIDO', '0') == '1'
EVAL_ON = os.getenv('BASELINE_EVAL_ON', 'dev').strip().lower()
# Cierre dev ensamble 2026-05: por defecto se reentrena para no reutilizar
# predicciones históricas con otro `max_length`. Usar TRF_REUSAR_PREDICCIONES=1
# solo para auditoría explícita, no para el cierre formal.
TRF_REUSAR_PREDICCIONES = os.getenv('TRF_REUSAR_PREDICCIONES', '0') == '1'
if EVAL_ON not in {'dev', 'test'}:
    raise ValueError(f"BASELINE_EVAL_ON inválido: {EVAL_ON}. Use 'dev' o 'test'.")
if EVAL_ON == 'test' and os.getenv('TRF_PERMITIR_TEST', '0') != '1':
    raise ValueError('TEST debe permanecer virgen. Para auditoría excepcional, definir TRF_PERMITIR_TEST=1 explícitamente.')
print('MODO_RAPIDO:', MODO_RAPIDO)
print('EVAL_ON:', EVAL_ON)
print('TRF_REUSAR_PREDICCIONES:', TRF_REUSAR_PREDICCIONES)

def resolve_device():
    import torch
    if torch.cuda.is_available():
        print('Usando CUDA')
        return torch.device('cuda')
    if torch.backends.mps.is_available():
        print('Usando MPS')
        return torch.device('mps')
    print('Usando CPU')
    return torch.device('cpu')

# Evita inicializar backend de entrenamiento cuando solo se reevalúa
if TRF_REUSAR_PREDICCIONES:
    device = 'reuso_predicciones'
    print('Modo reuso activo: se reusan predicciones si existen; si no, se inicializa entrenamiento con el dispositivo disponible.')
else:
    device = resolve_device()


MODO_RAPIDO: False
EVAL_ON: dev
TRF_REUSAR_PREDICCIONES: True
Modo reuso activo: se reusan predicciones si existen; si no, se inicializa entrenamiento con el dispositivo disponible.


In [2]:
# Hiperparámetros
# Cierre dev ensamble 2026-05: `max_length` queda fijo en 512 para todos
# los Transformers standalone. Esto alinea la rama contextual del ensamble
# con la condición donde el ensamble obtuvo el mejor resultado en dev.
# Las corridas previas con 256 se conservan como sensibilidad no adoptada.
MAX_LENGTH = 512
BATCH_SIZE = 2 if MODO_RAPIDO else 4
ACCUMULATION_STEPS = 2 if MODO_RAPIDO else 4
EPOCHS = 1 if MODO_RAPIDO else 3
LEARNING_RATE = 2e-5

MODELOS = {
    'beto': 'dccuchile/bert-base-spanish-wwm-cased',
    'roberta_biomedical': 'PlanTL-GOB-ES/roberta-base-biomedical-es',
    'roberta_clinical': 'PlanTL-GOB-ES/roberta-base-biomedical-clinical-es',
}
if MODO_RAPIDO:
    MODELOS = {'beto': MODELOS['beto']}

print('MAX_LENGTH:', MAX_LENGTH)
print('BATCH_SIZE:', BATCH_SIZE)
print('EPOCHS:', EPOCHS)
print('Modelos:', list(MODELOS.keys()))

MODEL_METADATA = {}
HISTORICAL_TRF_DIR = OUTPUTS_PATH / 'transformers_historicos'

def registrar_model_metadata(model_name: str, model_id: str, model=None) -> dict:
    """Registra configuración mínima para no confundir hidden_size con max_length."""
    hidden_size = None
    model_type = None
    if model is not None:
        hidden_size = getattr(model.config, 'hidden_size', None)
        model_type = getattr(model.config, 'model_type', None)
    else:
        try:
            from transformers import AutoConfig
            cfg = AutoConfig.from_pretrained(model_id)
            hidden_size = getattr(cfg, 'hidden_size', None)
            model_type = getattr(cfg, 'model_type', None)
        except Exception as e:
            print(f'Aviso: no se pudo leer config de {model_name}: {e}')
    meta = {
        'modelo': model_name,
        'model_id': model_id,
        'model_type': model_type,
        'hidden_size': int(hidden_size) if hidden_size is not None else None,
        'max_length': MAX_LENGTH,
        'representacion': 'contextual_densa',
        'salida_ensamble': 'prob_ansiedad|prob_depresion',
    }
    MODEL_METADATA[model_name] = meta
    return meta

def preservar_artefactos_historicos_si_corresponde(model_name: str):
    """Antes de sobrescribir salidas globales, copia artefactos con otro max_length."""
    eval_path = DATA_PATH / f'{model_name}_eval.csv'
    if not eval_path.exists():
        return None
    try:
        prev_eval = pd.read_csv(eval_path)
        prev_max = int(pd.to_numeric(prev_eval.get('max_length'), errors='coerce').dropna().iloc[-1])
    except Exception:
        return None
    if prev_max == MAX_LENGTH:
        return None

    hist_dir = HISTORICAL_TRF_DIR / f'{model_name}_max_length_{prev_max}'
    hist_dir.mkdir(parents=True, exist_ok=True)
    candidatos = [
        eval_path,
        DATA_PATH / f'{model_name}_classification_report.csv',
        DATA_PATH / f'{model_name}_predicciones_{EVAL_ON}.csv',
        DATA_PATH / f'{model_name}_predicciones_dev.csv',
    ]
    copiados = []
    for src in candidatos:
        if src.exists():
            dst = hist_dir / src.name
            shutil.copy2(src, dst)
            copiados.append(str(dst))
    manifest = {
        'modelo': model_name,
        'max_length_historico': prev_max,
        'max_length_nuevo': MAX_LENGTH,
        'motivo': 'preservar corrida historica antes de cierre dev alineado',
        'archivos_copiados': copiados,
    }
    with open(hist_dir / 'manifest_historico.json', 'w', encoding='utf-8') as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)
    print(f'Artefactos históricos preservados para {model_name}: {hist_dir}')
    return manifest


MAX_LENGTH: 512
BATCH_SIZE: 4
EPOCHS: 3
Modelos: ['beto', 'roberta_biomedical', 'roberta_clinical']


In [3]:
# Carga de datos
def _load_split_denoised(split_name: str) -> pd.DataFrame:
    p = SPLITS_PATH / f'{split_name}_denoised.csv'
    if not p.exists():
        raise FileNotFoundError(f'No existe {p}. Ejecuta 03_denoising_reglas_core primero.')
    return pd.read_csv(p)

df_train = _load_split_denoised('train')
df_eval = _load_split_denoised(EVAL_ON)

if MODO_RAPIDO:
    df_train = df_train.sample(min(len(df_train), 300), random_state=42)
    df_eval = df_eval.sample(min(len(df_eval), 150), random_state=42)

# Etiquetas detectadas en el corte actual (binaria)
all_labels = sorted(set(df_train['etiqueta'].astype(str)) | set(df_eval['etiqueta'].astype(str)))
label2id = {lab: i for i, lab in enumerate(all_labels)}
id2label = {i: lab for lab, i in label2id.items()}

for df in (df_train, df_eval):
    df['label'] = df['etiqueta'].astype(str).map(label2id)

print('Clases detectadas:', all_labels)
print('Train:', len(df_train), '| Eval:', len(df_eval), f'| split={EVAL_ON}')


Clases detectadas: ['ansiedad', 'depresion']
Train: 1107 | Eval: 343 | split=dev


In [4]:
# Limpieza textual conservadora
RE_MULTI = re.compile(r'(.){2,}')

def clean_text_trf(s: str) -> str:
    if pd.isna(s):
        return ''
    s = str(s).strip()
    s = unicodedata.normalize('NFC', s)
    s = RE_MULTI.sub(r'', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

df_train['texto_trf'] = df_train['texto'].map(clean_text_trf)
df_eval['texto_trf'] = df_eval['texto'].map(clean_text_trf)


In [5]:
import inspect

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    pred_ids = np.argmax(predictions, axis=1)
    labels_str = [id2label[int(x)] for x in labels]
    preds_str = [id2label[int(x)] for x in pred_ids]
    m = calculate_metrics(labels_str, preds_str)
    return {
        'f1': m['f1_macro'],
        'precision': m['precision_macro'],
        'recall': m['recall_macro'],
        'accuracy': m['accuracy'],
    }

rows = []
eval_row_ids = set(df_eval['row_id'].astype(int).tolist())

def exportar_metricas_desde_predicciones(model_name: str, model_id: str, pred_df: pd.DataFrame):
    cols_req = {'row_id', 'y_true', 'y_pred'}
    if not cols_req.issubset(pred_df.columns):
        raise ValueError(f'Predicciones de {model_name} sin columnas requeridas: {cols_req}')

    # En cierre formal no se deben reusar predicciones generadas con otro max_length.
    prev_eval_path = DATA_PATH / f'{model_name}_eval.csv'
    if prev_eval_path.exists():
        prev_eval = pd.read_csv(prev_eval_path)
        prev_max = pd.to_numeric(prev_eval.get('max_length'), errors='coerce').dropna()
        if not prev_max.empty and int(prev_max.iloc[-1]) != MAX_LENGTH:
            raise ValueError(
                f'Predicciones previas de {model_name} corresponden a max_length={int(prev_max.iloc[-1])}; '
                f'el cierre formal requiere max_length={MAX_LENGTH}.'
            )

    pred_df = pred_df.copy()
    pred_df['row_id'] = pred_df['row_id'].astype(int)
    pred_df = pred_df[pred_df['row_id'].isin(eval_row_ids)].copy()
    if pred_df.empty:
        raise ValueError(f'Predicciones de {model_name} sin intersección con {EVAL_ON}_denoised.')

    true_labels = pred_df['y_true'].astype(str).tolist()
    pred_labels = pred_df['y_pred'].astype(str).tolist()
    m = calculate_metrics(true_labels, pred_labels)

    pred_path = DATA_PATH / f'{model_name}_predicciones_{EVAL_ON}.csv'
    pred_df.to_csv(pred_path, index=False)

    report_path = DATA_PATH / f'{model_name}_classification_report.csv'
    report_df = pd.DataFrame(m['report_dict']).transpose()
    report_df.to_csv(report_path)

    eval_path = DATA_PATH / f'{model_name}_eval.csv'
    meta = MODEL_METADATA.get(model_name) or registrar_model_metadata(model_name, model_id)
    metrics_df = pd.DataFrame([{
        'modelo': model_name,
        'model_id': meta.get('model_id'),
        'model_type': meta.get('model_type'),
        'hidden_size': meta.get('hidden_size'),
        'representacion': meta.get('representacion'),
        'salida_ensamble': meta.get('salida_ensamble'),
        'f1_macro': m['f1_macro'],
        'precision_macro': m['precision_macro'],
        'recall_macro': m['recall_macro'],
        'accuracy': m['accuracy'],
        'n_train': len(df_train),
        'n_eval': len(pred_df),
        'n_dev': len(pred_df),  # compatibilidad hacia atrás
        'eval_split': EVAL_ON,
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'max_length': MAX_LENGTH,
        'n_clases': len(all_labels),
        'clases': '|'.join(all_labels),
        'device': str(device),
    }])
    metrics_df.to_csv(eval_path, index=False)

    rows.append(metrics_df.iloc[0].to_dict())
    print(f'Exportado {eval_path.name}, {report_path.name}, {pred_path.name}')


# El archivo `<modelo>_eval.csv` se utiliza en 08 para comparar líneas base.
for model_name, model_id in MODELOS.items():
    print(f"\n=== Procesando {model_name} ===")
    registrar_model_metadata(model_name, model_id)
    preservar_artefactos_historicos_si_corresponde(model_name)

    # Camino rápido: reusar predicciones existentes y recalcular métricas sobre el split denoised
    if TRF_REUSAR_PREDICCIONES:
        pred_candidatos = [
            DATA_PATH / f'{model_name}_predicciones_{EVAL_ON}.csv',
            DATA_PATH / f'{model_name}_predicciones_dev.csv',
        ]
        for p in pred_candidatos:
            if not p.exists():
                continue
            try:
                pred_df = pd.read_csv(p)
                exportar_metricas_desde_predicciones(model_name, model_id, pred_df)
                print(f'Reutilizado: {p.name}')
                break
            except Exception as e:
                print(f'Aviso: no se pudo reutilizar {p.name}: {e}')
        else:
            print('No hay predicciones previas reutilizables; se entrena modelo.')
            pred_df = None

        if pred_df is not None and not pred_df.empty:
            continue

    print(f"Entrenando {model_name} desde cero...")
    if device == 'reuso_predicciones':
        device = resolve_device()

    from datasets import Dataset
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

    ds_train = Dataset.from_pandas(df_train[['row_id', 'texto_trf', 'label']].rename(columns={'texto_trf': 'texto'}))
    ds_eval = Dataset.from_pandas(df_eval[['row_id', 'texto_trf', 'label']].rename(columns={'texto_trf': 'texto'}))

    tokenizer = AutoTokenizer.from_pretrained(model_id)

    def tokenize_function(examples):
        return tokenizer(
            examples['texto'],
            padding='max_length',
            truncation=True,
            max_length=MAX_LENGTH,
        )

    tokenized_train = ds_train.map(tokenize_function, batched=True)
    tokenized_eval = ds_eval.map(tokenize_function, batched=True)

    import torch

    model = AutoModelForSequenceClassification.from_pretrained(
        model_id,
        num_labels=len(all_labels),
        id2label=id2label,
        label2id=label2id,
    )
    model = model.to(device)
    meta = registrar_model_metadata(model_name, model_id, model=model)

    out_ckpt = CHECKPOINTS_PATH / model_name
    out_log = LOGS_PATH / model_name
    out_ckpt.mkdir(parents=True, exist_ok=True)
    out_log.mkdir(parents=True, exist_ok=True)

    ta_params = inspect.signature(TrainingArguments.__init__).parameters
    strategy_key = 'evaluation_strategy' if 'evaluation_strategy' in ta_params else 'eval_strategy'

    ta_kwargs = {
        'output_dir': str(out_ckpt),
        'learning_rate': LEARNING_RATE,
        'per_device_train_batch_size': BATCH_SIZE,
        'per_device_eval_batch_size': BATCH_SIZE,
        'gradient_accumulation_steps': ACCUMULATION_STEPS,
        'gradient_checkpointing': True,
        'num_train_epochs': EPOCHS,
        'weight_decay': 0.01,
        strategy_key: 'epoch',
        'save_strategy': 'epoch',
        'load_best_model_at_end': True,
        'metric_for_best_model': 'f1',
        'logging_dir': str(out_log),
        'logging_steps': 10,
        'seed': 42,
        'report_to': 'none',
    }
    args = TrainingArguments(**ta_kwargs)

    trainer_kwargs = {
        'model': model,
        'args': args,
        'train_dataset': tokenized_train,
        'eval_dataset': tokenized_eval,
        'compute_metrics': compute_metrics,
    }
    trainer_params = inspect.signature(Trainer.__init__).parameters
    if 'tokenizer' in trainer_params:
        trainer_kwargs['tokenizer'] = tokenizer
    else:
        trainer_kwargs['processing_class'] = tokenizer

    trainer = Trainer(**trainer_kwargs)

    trainer.train()
    eval_results = trainer.evaluate()

    # Predicciones para auditoría
    pred_out = trainer.predict(tokenized_eval)
    pred_ids = np.argmax(pred_out.predictions, axis=1)
    pred_labels = [id2label[int(x)] for x in pred_ids]
    true_labels = [id2label[int(x)] for x in pred_out.label_ids]

    pred_df = pd.DataFrame({
        'row_id': ds_eval['row_id'],
        'y_true': true_labels,
        'y_pred': pred_labels,
    })

    # Probabilidades por clase
    probs = torch.nn.functional.softmax(torch.tensor(pred_out.predictions), dim=1).numpy()
    for i, lab in id2label.items():
        pred_df[f'prob_{lab}'] = probs[:, int(i)]

    pred_path = DATA_PATH / f'{model_name}_predicciones_{EVAL_ON}.csv'
    pred_df.to_csv(pred_path, index=False)

    report_path = DATA_PATH / f'{model_name}_classification_report.csv'
    report_df = pd.DataFrame(calculate_metrics(true_labels, pred_labels)['report_dict']).transpose()
    report_df.to_csv(report_path)

    eval_path = DATA_PATH / f'{model_name}_eval.csv'
    metrics_df = pd.DataFrame([{
        'modelo': model_name,
        'model_id': meta.get('model_id'),
        'model_type': meta.get('model_type'),
        'hidden_size': meta.get('hidden_size'),
        'representacion': meta.get('representacion'),
        'salida_ensamble': meta.get('salida_ensamble'),
        'f1_macro': eval_results['eval_f1'],
        'precision_macro': eval_results['eval_precision'],
        'recall_macro': eval_results['eval_recall'],
        'accuracy': eval_results['eval_accuracy'],
        'n_train': len(df_train),
        'n_eval': len(df_eval),
        'n_dev': len(df_eval),  # compatibilidad hacia atrás
        'eval_split': EVAL_ON,
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'max_length': MAX_LENGTH,
        'n_clases': len(all_labels),
        'clases': '|'.join(all_labels),
        'device': str(device),
    }])
    metrics_df.to_csv(eval_path, index=False)

    rows.append(metrics_df.iloc[0].to_dict())
    print(f'Exportado {eval_path.name}, {report_path.name}, {pred_path.name}')

    del model, trainer, tokenized_train, tokenized_eval
    if hasattr(device, 'type') and device.type == 'cuda':
        torch.cuda.empty_cache()
    elif hasattr(device, 'type') and device.type == 'mps':
        torch.mps.empty_cache()



=== Procesando beto ===
Exportado beto_eval.csv, beto_classification_report.csv, beto_predicciones_dev.csv
Reutilizado: beto_predicciones_dev.csv

=== Procesando roberta_biomedical ===
Exportado roberta_biomedical_eval.csv, roberta_biomedical_classification_report.csv, roberta_biomedical_predicciones_dev.csv
Reutilizado: roberta_biomedical_predicciones_dev.csv

=== Procesando roberta_clinical ===
Exportado roberta_clinical_eval.csv, roberta_clinical_classification_report.csv, roberta_clinical_predicciones_dev.csv
Reutilizado: roberta_clinical_predicciones_dev.csv


In [6]:
df_resumen = pd.DataFrame(rows)
if df_resumen.empty:
    raise ValueError('No se generaron resultados Transformer en 04c.')

# Normalización mínima para comparación homogénea
for col in ['f1_macro', 'precision_macro', 'recall_macro', 'accuracy']:
    if col in df_resumen.columns:
        df_resumen[col] = pd.to_numeric(df_resumen[col], errors='coerce')

if 'balanced_acc' not in df_resumen.columns:
    if 'recall_macro' in df_resumen.columns:
        df_resumen['balanced_acc'] = df_resumen['recall_macro']
    else:
        df_resumen['balanced_acc'] = np.nan

df_resumen['eval_split'] = df_resumen.get('eval_split', EVAL_ON)
df_resumen['modelo'] = df_resumen['modelo'].astype(str)
df_rank = df_resumen.sort_values(['f1_macro', 'balanced_acc', 'precision_macro'], ascending=False).reset_index(drop=True)
best = df_rank.iloc[0].to_dict()

ts = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
selection_run_id = f'trfsel_{ts}'
comparativa_csv = OUTPUTS_PATH / f'transformer_baseline_comparison_{ts}.csv'
selection_json = OUTPUTS_PATH / f'transformer_baseline_selection_{ts}.json'
latest_json = OUTPUTS_PATH / 'transformer_baseline_selection_latest.json'

df_rank.to_csv(comparativa_csv, index=False)

cols_modelos_comparados = [
    'modelo', 'model_id', 'model_type', 'hidden_size', 'representacion',
    'salida_ensamble', 'max_length', 'f1_macro', 'balanced_acc',
    'precision_macro', 'recall_macro', 'accuracy', 'n_train', 'n_eval'
]
cols_modelos_comparados = [c for c in cols_modelos_comparados if c in df_rank.columns]

payload = {
    'run_id': selection_run_id,
    'fecha': pd.Timestamp.now().isoformat(),
    'eval_split': EVAL_ON,
    'criterio_seleccion': 'max(f1_macro), desempate por balanced_acc y precision_macro',
    'modelos_comparados': df_rank[cols_modelos_comparados].to_dict(orient='records'),
    'mejor_transformer_baseline': {
        'modelo': str(best.get('modelo')),
        'f1_macro': float(best.get('f1_macro')) if pd.notna(best.get('f1_macro')) else None,
        'balanced_acc': float(best.get('balanced_acc')) if pd.notna(best.get('balanced_acc')) else None,
        'precision_macro': float(best.get('precision_macro')) if pd.notna(best.get('precision_macro')) else None,
        'recall_macro': float(best.get('recall_macro')) if pd.notna(best.get('recall_macro')) else None,
    },
    'observaciones': [
        'Selección realizada solo en split de desarrollo.',
        'Este artefacto se usa como entrada por defecto para escoger backbone contextual en 06.',
    ],
    'paths': {
        'comparativa_csv': str(comparativa_csv),
        'selection_json': str(selection_json),
        'latest_json': str(latest_json),
    },
}

with open(selection_json, 'w', encoding='utf-8') as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)
with open(latest_json, 'w', encoding='utf-8') as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

display(df_rank)
print('Proceso baseline Transformers finalizado.')
print('Comparativa exportada:', comparativa_csv)
print('Selección exportada:', selection_json)
print('Selección latest:', latest_json)
print('Mejor baseline Transformer en', EVAL_ON, ':', payload['mejor_transformer_baseline']['modelo'])


,modelo,model_id,model_type,hidden_size,representacion,salida_ensamble,f1_macro,precision_macro,recall_macro,accuracy,...,n_eval,n_dev,eval_split,epochs,batch_size,max_length,n_clases,clases,device,balanced_acc
0,roberta_clinical,PlanTL-GOB-ES/roberta-base-biomedical-clinical-es,roberta,768,contextual_densa,prob_ansiedad|prob_depresion,0.741078,0.732286,0.763889,0.769679,...,343,343,dev,3,4,512,2,ansiedad|depresion,reuso_predicciones,0.763889
1,beto,dccuchile/bert-base-spanish-wwm-cased,bert,768,contextual_densa,prob_ansiedad|prob_depresion,0.735052,0.727153,0.749177,0.769679,...,343,343,dev,3,4,512,2,ansiedad|depresion,reuso_predicciones,0.749177
2,roberta_biomedical,PlanTL-GOB-ES/roberta-base-biomedical-es,roberta,768,contextual_densa,prob_ansiedad|prob_depresion,0.722794,0.717362,0.730350,0.763848,...,343,343,dev,3,4,512,2,ansiedad|depresion,reuso_predicciones,0.730350


Proceso baseline Transformers finalizado.
Comparativa exportada: /Users/manuelnunez/Projects/psych-phenotyping-paraguay/data/outputs/transformer_baseline_comparison_20260512_155439.csv
Selección exportada: /Users/manuelnunez/Projects/psych-phenotyping-paraguay/data/outputs/transformer_baseline_selection_20260512_155439.json
Selección latest: /Users/manuelnunez/Projects/psych-phenotyping-paraguay/data/outputs/transformer_baseline_selection_latest.json
Mejor baseline Transformer en dev : roberta_clinical
